**Sentrimix**

Kaggle Dataset - https://www.kaggle.com/datasets/aaron7sun/stocknews

*First, upload Combined_News_DJIA.csv from the link above into Colab.*

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("Combined_News_DJIA.csv")

display(df.head())

In [ ]:
import ast
import pandas as pd

def clean_headline(value):
    if pd.isna(value):
        return ""

    # Convert string representations of byte strings into text
    if isinstance(value, str) and value.startswith(("b'", 'b"')):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            pass

    # Decode actual byte strings
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")

    return str(value).strip()

# Identify all headline columns
headline_columns = [f"Top{i}" for i in range(1, 26)]

# Clean every headline
for col in headline_columns:
    df[col] = df[col].apply(clean_headline)

display(df[["Date", "Label", "Top1", "Top2", "Top3"]].head())

In [ ]:
print("Dataset shape:", df.shape)
print("Missing values:", df[headline_columns].isnull().sum().sum())

print("\nExample cleaned headline:")
print(df["Top1"].iloc[0])

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

# Sort chronologically just to be safe
df = df.sort_values("Date").reset_index(drop=True)

In [ ]:
#Split the dataset into train and test by the dates specified.
train_df = df[
    (df["Date"] >= "2008-01-01") &
    (df["Date"] < "2015-01-01")
].copy()

test_df = df[
    (df["Date"] >= "2015-01-01") &
    (df["Date"] < "2017-01-01")
].copy()

In [ ]:
print("Training set:")
print("Rows:", len(train_df))
print("Date range:", train_df["Date"].min(), "to", train_df["Date"].max())

print("\nTesting set:")
print("Rows:", len(test_df))
print("Date range:", test_df["Date"].min(), "to", test_df["Date"].max())

In [ ]:
print("Training label distribution:")
print(train_df["Label"].value_counts(normalize=True))

print("\nTesting label distribution:")
print(test_df["Label"].value_counts(normalize=True))

In [ ]:
# SENTRIMIX - FINBERT SENTIMENT CLASSIFICATION

!pip install -q transformers tqdm

import pandas as pd
import numpy as np
import torch
import os

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("GPU unavailable. Using CPU.")

model_name = "ProsusAI/finbert"

print("\nLoading FinBERT...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

finbert = AutoModelForSequenceClassification.from_pretrained(
    model_name
)

finbert.to(device)
finbert.eval()

label_map = {
    label.lower(): int(index)
    for index, label in finbert.config.id2label.items()
}

print("FinBERT successfully loaded!")
print("Sentiment labels:", label_map)

# PREPARE TRAINING AND TESTING DATA

train_copy = train_df.copy()
test_copy = test_df.copy()

train_copy["Split"] = "train"
test_copy["Split"] = "test"

combined_df = pd.concat(
    [train_copy, test_copy],
    ignore_index=True
)

headline_columns = [f"Top{i}" for i in range(1, 26)]

headlines_df = combined_df.melt(
    id_vars=["Date", "Label", "Split"],
    value_vars=headline_columns,
    var_name="Headline_Number",
    value_name="Headline"
)

headlines_df["Headline"] = (
    headlines_df["Headline"]
    .fillna("")
    .astype(str)
    .str.strip()
)

headlines_df = headlines_df[
    headlines_df["Headline"] != ""
].reset_index(drop=True)

print("\nTotal headlines:", len(headlines_df))

# CLASSIFY HEADLINES IN BATCHES

batch_size = 32 if device.type == "cuda" else 16

output_file = "SENTRIMIX_FinBERT_Results.csv"

# Fresh run: remove any old output
if os.path.exists(output_file):
    os.remove(output_file)

results = []

print("\nStarting sentiment classification...")

for start in tqdm(
    range(0, len(headlines_df), batch_size),
    desc="Classifying headlines"
):

    batch_df = headlines_df.iloc[start:start + batch_size]
    batch = batch_df["Headline"].tolist()

    inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = finbert(**inputs)
        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

    probabilities = probabilities.cpu().numpy()

    for i, probs in enumerate(probabilities):

        positive = float(probs[label_map["positive"]])
        negative = float(probs[label_map["negative"]])
        neutral = float(probs[label_map["neutral"]])

        scores = {
            "positive": positive,
            "negative": negative,
            "neutral": neutral
        }

        results.append({
            "Date": batch_df.iloc[i]["Date"],
            "Label": batch_df.iloc[i]["Label"],
            "Split": batch_df.iloc[i]["Split"],
            "Headline_Number": batch_df.iloc[i]["Headline_Number"],
            "Headline": batch[i],
            "Sentiment": max(scores, key=scores.get),
            "Positive_Score": positive,
            "Negative_Score": negative,
            "Neutral_Score": neutral
        })

    # Save progress every 1,000 or so headlines
    if len(results) >= 1000:
        pd.DataFrame(results).to_csv(
            output_file,
            mode="a",
            header=not os.path.exists(output_file),
            index=False
        )
        results = []

# Save remaining results
if results:
    pd.DataFrame(results).to_csv(
        output_file,
        mode="a",
        header=not os.path.exists(output_file),
        index=False
    )

# LOAD AND INSPECT RESULTS

sentiment_df = pd.read_csv(output_file)

print("\n======================================")
print("SENTRIMIX CLASSIFICATION COMPLETE")
print("======================================")

print("Total classified headlines:", len(sentiment_df))

print("\nSentiment distribution:")
print(sentiment_df["Sentiment"].value_counts())

print("\nExample classifications:")
display(
    sentiment_df[
        ["Date", "Headline", "Sentiment",
         "Positive_Score", "Negative_Score", "Neutral_Score"]
    ].head(10)
)

# Output's FinBert's scoring for neutrality, positivity and negativity per headline

from google.colab import files

files.download(output_file)


We now have the CSV which has each headline and FinBert's score for neutrality, positivity and negativity. We now make it into one row per day.

In [ ]:

# SENTRIMIX: Daily Sentiment Aggregation

import pandas as pd

# Load FinBERT classification results
sentiment_df = pd.read_csv("SENTRIMIX_FinBERT_Results.csv")

sentiment_df["Date"] = pd.to_datetime(sentiment_df["Date"])

# Aggregate headline sentiment into daily features
daily_features = sentiment_df.groupby(
    ["Date", "Label", "Split"]
).agg(
    Positive_Mean=("Positive_Score", "mean"),
    Negative_Mean=("Negative_Score", "mean"),
    Neutral_Mean=("Neutral_Score", "mean"),

    Positive_Ratio=("Sentiment", lambda x: (x == "positive").mean()),
    Negative_Ratio=("Sentiment", lambda x: (x == "negative").mean()),
    Neutral_Ratio=("Sentiment", lambda x: (x == "neutral").mean()),

    Headline_Count=("Headline", "count")
).reset_index()

# Calculate net sentiment
daily_features["Net_Sentiment"] = (
    daily_features["Positive_Mean"]
    - daily_features["Negative_Mean"]
)

# Sort chronologically
daily_features = daily_features.sort_values("Date").reset_index(drop=True)

# Separate training and testing features
daily_train = daily_features[
    daily_features["Split"] == "train"
].copy()

daily_test = daily_features[
    daily_features["Split"] == "test"
].copy()

# Save results
daily_features.to_csv(
    "SENTRIMIX_Daily_Features.csv",
    index=False
)

# Inspect results
print("Daily aggregation complete!")
print("Training days:", len(daily_train))
print("Testing days:", len(daily_test))

display(daily_features.head())

# Download daily features
from google.colab import files
files.download("SENTRIMIX_Daily_Features.csv")

In [ ]:
# Upload Daily Features which you would have downloaded from last sell

from google.colab import files
import io
import pandas as pd

uploaded = files.upload()

# Load the uploaded CSV
df = pd.read_csv(io.BytesIO(uploaded["SENTRIMIX_Daily_Features.csv"]))

print("Daily features successfully loaded!")
print("Dataset shape:", df.shape)

display(df.head())


In [ ]:

# SENTRIMIX: Logistic Regression Model

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load the daily sentiment features
df = pd.read_csv("SENTRIMIX_Daily_Features.csv")

# 2. Select sentiment features for the model
features = [
    "Positive_Mean",
    "Negative_Mean",
    "Neutral_Mean",
    "Positive_Ratio",
    "Negative_Ratio",
    "Neutral_Ratio",
    "Net_Sentiment"
]

# 3. Separate training and testing data
train = df[df["Split"] == "train"].copy()
test = df[df["Split"] == "test"].copy()

# 4. Separate inputs (X) and target labels (y)
X_train = train[features]
y_train = train["Label"]

X_test = test[features]
y_test = test["Label"]

# 5. Standardise features using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Create and train the Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 7. Predict market direction on unseen test data
predictions = model.predict(X_test_scaled)

# 8. Evaluate model performance
accuracy = accuracy_score(y_test, predictions)

print("SENTRIMIX LOGISTIC REGRESSION RESULTS")
print("-------------------------------------")
print(f"Training records: {len(X_train)}")
print(f"Testing records: {len(X_test)}")
print(f"Test accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Here, we observe heavy class imbalance, so we begin to use Scikit learn's balanced accuracy for better understanding on how the model performs in prediction in both directions.

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

# Split the training data chronologically:
# First 80% for training, final 20% for validation
X = train[features]
y = train["Label"]

split_point = int(len(X) * 0.8)

X_inner_train = X.iloc[:split_point]
X_val = X.iloc[split_point:]

y_inner_train = y.iloc[:split_point]
y_val = y.iloc[split_point:]

# Scale using inner training data only
scaler = StandardScaler()
X_inner_scaled = scaler.fit_transform(X_inner_train)
X_val_scaled = scaler.transform(X_val)

# Compare standard and balanced Logistic Regression
for weighting in [None, "balanced"]:

    model = LogisticRegression(
        max_iter=1000,
        class_weight=weighting
    )

    model.fit(X_inner_scaled, y_inner_train)
    predictions = model.predict(X_val_scaled)

    print("\nClass weighting:", weighting)
    print("Accuracy:", round(accuracy_score(y_val, predictions) * 100, 2), "%")
    print("Balanced accuracy:", round(balanced_accuracy_score(y_val, predictions) * 100, 2), "%")
    print("Confusion matrix:")
    print(confusion_matrix(y_val, predictions))


In [ ]:
#Punishes the model's weights using regularisation, to reduce variance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Test different regularisation strengths
C_values = [0.01, 0.1, 1, 10, 100]

for c in C_values:

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=c
    )

    model.fit(X_inner_scaled, y_inner_train)
    predictions = model.predict(X_val_scaled)

    print(f"\nC = {c}")
    print("Accuracy:", round(accuracy_score(y_val, predictions) * 100, 2), "%")
    print("Balanced accuracy:", round(balanced_accuracy_score(y_val, predictions) * 100, 2), "%")


In [ ]:

# SENTRIMIX: Final Logistic Regression Evaluation

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Scale using the complete training dataset
final_scaler = StandardScaler()

X_train_final = final_scaler.fit_transform(X_train)
X_test_final = final_scaler.transform(X_test)

# Train selected model
final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    C=1
)

final_model.fit(X_train_final, y_train)

# Evaluate on untouched test data
final_predictions = final_model.predict(X_test_final)

print("FINAL SENTRIMIX RESULTS")
print("-----------------------")
print("Accuracy:",
      round(accuracy_score(y_test, final_predictions) * 100, 2), "%")

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, final_predictions) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, final_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_predictions))

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate confusion matrix
cm = confusion_matrix(y_test, predictions)

# Plot heatmap
plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=1,
    linecolor="white",
    xticklabels=["Fall (0)", "Rise / Unchanged (1)"],
    yticklabels=["Fall (0)", "Rise / Unchanged (1)"],
    annot_kws={"size": 16}
)

plt.title("SENTRIMIX Confusion Matrix", fontsize=15, pad=15)
plt.xlabel("Predicted Outcome", fontsize=12)
plt.ylabel("Actual Outcome", fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:

# SENTRIMIX: Baseline Comparisons

import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Baseline 1: Always predict UP (Label 1)
always_up_predictions = np.ones(len(y_test), dtype=int)

# Baseline 2: Predict previous trading day's direction
all_labels = df.sort_values("Date")["Label"].reset_index(drop=True)

test_indices = df.sort_values("Date").reset_index(drop=True).index[
    df.sort_values("Date").reset_index(drop=True)["Split"] == "test"
]

previous_day_predictions = all_labels.shift(1).iloc[test_indices].to_numpy()

# Evaluate both baselines
baselines = {
    "Always Predict UP": always_up_predictions,
    "Previous Day Direction": previous_day_predictions
}

print("SENTRIMIX: BASELINE COMPARISON")
print("--------------------------------")

for name, predictions in baselines.items():

    accuracy = accuracy_score(y_test, predictions)
    balanced_accuracy = balanced_accuracy_score(y_test, predictions)

    print(f"\n{name}")
    print("Accuracy:", round(accuracy * 100, 2), "%")
    print("Balanced accuracy:", round(balanced_accuracy * 100, 2), "%")

# Compare against SENTRIMIX
print("\nFINAL MODEL")
print("Accuracy:",
      round(accuracy_score(y_test, final_predictions) * 100, 2), "%")
print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, final_predictions) * 100, 2), "%")


In [ ]:

!pip install -q joblib

import joblib
from google.colab import files

joblib.dump(
    {
        "model": model,
        "scaler": scaler,
        "features": features
    },
    "SENTRIMIX_Logistic_Regression.joblib"
)

files.download("SENTRIMIX_Logistic_Regression.joblib")

#Download the model so that we can use it in our application without rerunning everything

Now, we have a model which has accuracy 52.65% compared to 50.79% of the UP baseline and 45.77% of the Previous Day Direction Baseline. I was on a severe crunch of time.

To improve I would have done the following:
1.   I would have collated my own data to compare the Sentiment Analysis API's predictions rather than just assuming that FinBert is the best by research.
2.   I would have plotted more graphs to observe training vs testing accuracy and understand whether my model was overfitting vs underfitting, but going by each epoch and getting these metrics would have taken too much time.

Unfortunately, I was working on a few things before running out of time:
*   Experimenting with the features which we feed to the model
*   Comparing my Logistic Regression model with Random Forest models, but this proved to not be as accurate.





In [ ]:
!pip install -q flask pyngrok transformers joblib

import os
import joblib
import torch
import pandas as pd
import numpy as np

from flask import Flask, request, render_template_string
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load your saved Logistic Regression model
saved = joblib.load(
    "/content/SENTRIMIX_Logistic_Regression.joblib"
)

model = saved["model"]
scaler = saved["scaler"]
features = saved["features"]

print("Logistic Regression loaded!")

# Load pretrained FinBERT
model_name = "ProsusAI/finbert"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

finbert = AutoModelForSequenceClassification.from_pretrained(
    model_name
)

finbert.to(device)
finbert.eval()

label_map = {
    label.lower(): int(index)
    for index, label in finbert.config.id2label.items()
}

print("FinBERT loaded!")
print("Device:", device)
print("SENTRIMIX models ready!")

In [ ]:
def predict_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        probs = torch.softmax(finbert(**inputs).logits, dim=1)[0].cpu().numpy()

    positive, negative, neutral = (float(probs[label_map[x]]) for x in ["positive", "negative", "neutral"])

    X = pd.DataFrame([{
        "Positive_Mean": positive,
        "Negative_Mean": negative,
        "Neutral_Mean": neutral,
        "Positive_Ratio": float(positive == max(positive, negative, neutral)),
        "Negative_Ratio": float(negative == max(positive, negative, neutral)),
        "Neutral_Ratio": float(neutral == max(positive, negative, neutral)),
        "Net_Sentiment": positive - negative
    }])[features]

    X = scaler.transform(X)
    p = model.predict_proba(X)[0]
    classes = list(model.classes_)

    return {
        "Prediction": "UP" if model.predict(X)[0] == 1 else "DOWN",
        "UP Probability": round(float(p[classes.index(1)]) * 100, 2),
        "DOWN Probability": round(float(p[classes.index(0)]) * 100, 2)
    }

predict_news("Microsoft reports strong quarterly earnings.")

In [ ]:
def predict_day(headlines):
    results = analyse_headlines(headlines)
    X = aggregate_features(results)[features]

    probabilities = model.predict_proba(
        scaler.transform(X)
    )[0]

    classes = list(model.classes_)
    up = float(probabilities[classes.index(1)]) * 100
    down = float(probabilities[classes.index(0)]) * 100

    if up < 45:
        label = "DOWN"
    elif up < 50:
        label = "MARGINALLY DOWN"
    elif up < 55:
        label = "MARGINALLY UP"
    else:
        label = "UP"

    return {
        "prediction": label,
        "up_probability": round(up, 2),
        "down_probability": round(down, 2),
        "headline_results": results
    }

In order to make a nice User Interface, I used Flask web framework with ngrok to run on a local development server.

In [ ]:

from flask import Flask, request, render_template_string
import pandas as pd
import torch
import numpy as np
import math

app = Flask(__name__)


HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
<title>SENTRIMIX</title>

<style>
* { box-sizing: border-box; }

body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: #0b1020;
    color: #edf2f7;
}

.header {
    text-align: center;
    padding: 55px 20px 30px;
}

.brand {
    color: #72e0c1;
    font-size: 14px;
    letter-spacing: 7px;
    font-weight: bold;
}

h1 { font-size: 44px; margin: 15px 0 8px; }

.subtitle {
    color: #aab8cf;
    letter-spacing: 3px;
    font-size: 13px;
}

.description {
    color: #aab8cf;
    margin-top: 25px;
    line-height: 1.7;
}

.container {
    max-width: 1050px;
    margin: auto;
    padding: 20px;
}

.card {
    background: #131d31;
    border: 1px solid #293853;
    border-radius: 16px;
    padding: 25px;
    margin-bottom: 22px;
}

.section-title {
    font-size: 19px;
    font-weight: bold;
    margin-bottom: 15px;
}

.muted {
    color: #aab8cf;
    font-size: 13px;
    line-height: 1.7;
}

textarea {
    width: 100%;
    height: 250px;
    background: #0b1427;
    border: 1px solid #34445f;
    border-radius: 10px;
    padding: 16px;
    color: white;
    font-size: 14px;
    resize: vertical;
}

button {
    width: 100%;
    padding: 16px;
    margin-top: 18px;
    background: #72e0c1;
    color: #10202a;
    border: none;
    border-radius: 9px;
    font-weight: bold;
    font-size: 15px;
    cursor: pointer;
}

button:hover { background: #9af0d8; }

.result {
    text-align: center;
    padding: 25px;
    border-radius: 12px;
    margin: 20px 0;
}

.result.up {
    background: #123b32;
    border: 1px solid #2e8068;
}

.result.marginal-up,
.result.marginal-down {
    background: #45371f;
    border: 1px solid #c18b37;
}

.result.down {
    background: #35151f;
    border: 1px solid #c52f4b;
}

.result h2 {
    font-size: 34px;
    margin: 8px;
}

.up-text { color: #72e0c1; }
.down-text { color: #ff6577; }

.probabilities {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 15px;
    margin: 20px 0;
}

.probability {
    background: #0b1427;
    padding: 18px;
    border-radius: 10px;
    text-align: center;
    border: 1px solid #293853;
}

.probability.down-prob {
    border-color: #a92f45;
    background: #24131d;
}

.probability h3 {
    font-size: 27px;
    margin: 8px;
}

.bar {
    height: 14px;
    display: flex;
    overflow: hidden;
    border-radius: 20px;
    background: #293853;
    margin: 15px 0;
}

.up-bar {
    background: #35b994;
    height: 100%;
}

.down-bar {
    background: #a92f45;
    height: 100%;
}

.model-score-box {
    background: #0b1427;
    border: 1px solid #34445f;
    border-radius: 12px;
    padding: 20px;
    text-align: center;
    margin-top: 18px;
}

.model-score-box h3 {
    font-size: 32px;
    margin: 10px;
}

.model-positive { color: #72e0c1; font-weight: bold; }
.model-negative { color: #ff6577; font-weight: bold; }
.model-amber { color: #f3bd60; font-weight: bold; }

.metrics {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 12px;
}

.metric {
    background: #0b1427;
    padding: 18px 10px;
    text-align: center;
    border-radius: 10px;
}

.metric h3 {
    color: #72e0c1;
    font-size: 24px;
    margin: 5px;
}

.metric p {
    color: #aab8cf;
    font-size: 12px;
}

details {
    background: #0b1427;
    border: 1px solid #293853;
    border-radius: 10px;
    padding: 16px;
    margin-top: 20px;
    color: #aab8cf;
    font-size: 13px;
    line-height: 1.7;
}

summary {
    color: #72e0c1;
    font-weight: bold;
    cursor: pointer;
}

details h4 {
    color: #edf2f7;
    margin-bottom: 5px;
}

.formula {
    color: #72e0c1;
    font-family: monospace;
    background: #131d31;
    padding: 10px;
    border-radius: 6px;
    margin: 10px 0;
    overflow-wrap: anywhere;
}

.net-summary {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 12px;
    margin-top: 20px;
}

.net-box {
    background: #0b1427;
    border: 1px solid #293853;
    border-radius: 10px;
    padding: 18px;
    text-align: center;
}

.net-box h3 {
    font-size: 25px;
    margin: 8px;
}

.net-box p {
    color: #aab8cf;
    font-size: 12px;
}

.net-positive { color: #72e0c1; font-weight: bold; }
.net-negative { color: #ff929e; font-weight: bold; }
.net-neutral { color: #aab8cf; font-weight: bold; }

.table-wrapper {
    width: 100%;
    overflow-x: auto;
}

table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
    min-width: 900px;
}

th, td {
    padding: 13px 10px;
    border-bottom: 1px solid #293853;
    text-align: left;
}

th { color: #72e0c1; }
td { color: #d4deed; }

.score { white-space: nowrap; }
.sentiment { font-weight: bold; }

.positive { color: #72e0c1; }
.negative { color: #ff929e; }
.neutral { color: #aab8cf; }

tfoot td {
    border-top: 2px solid #72e0c1;
    font-weight: bold;
    background: #0b1427;
}

.footer {
    text-align: center;
    padding: 30px;
    color: #77869f;
    font-size: 12px;
    line-height: 1.8;
}

.error {
    color: #ff9a9a;
    background: #42252d;
    padding: 15px;
    border-radius: 10px;
}

@media(max-width: 650px) {
    h1 { font-size: 32px; }

    .metrics,
    .probabilities,
    .net-summary {
        grid-template-columns: 1fr;
    }

    .card { padding: 17px; }
}
</style>
</head>

<body>

<div class="header">
    <div class="brand">S E N T R I M I X</div>
    <h1>Beyond the Headlines.</h1>
    <div class="subtitle">INTELLIGENCE BEYOND THE HEADLINES</div>

    <p class="description">
        Discover how financial news sentiment relates to
        market direction through artificial intelligence.
    </p>
</div>

<div class="container">

    <div class="card">
        <div class="section-title">Analyse your headlines</div>

        <p class="muted">
            Enter multiple headlines, one per line.
            Around 25 headlines best matches the original
            training data.
        </p>

        <form action="/analyse" method="POST">
            <textarea name="headlines"
                placeholder="Enter your headlines here..."
                required>{{ submitted or "" }}</textarea>

            <button type="submit">ANALYSE HEADLINES</button>
        </form>
    </div>

    {% if error %}
        <div class="error">{{ error }}</div>
    {% endif %}

    {% if result %}

    <div class="card">

        <div class="section-title">
            SENTRIMIX Market Prediction
        </div>

        <div class="result {{ result.direction_class }}">

            <p style="color:#aab8cf;">
                OVERALL PREDICTED MARKET DIRECTION
            </p>

            <h2>{{ result.direction }}</h2>

            <p>{{ result.explanation }}</p>
        </div>

        <div class="probabilities">

            <div class="probability">
                <p class="up-text">UP PROBABILITY</p>
                <h3 class="up-text">{{ result.up }}%</h3>
            </div>

            <div class="probability down-prob">
                <p class="down-text">DOWN PROBABILITY</p>
                <h3 class="down-text">{{ result.down }}%</h3>
            </div>

        </div>

        <div class="bar">
            <div class="up-bar"
                 style="width:{{ result.up }}%"></div>

            <div class="down-bar"
                 style="width:{{ result.down }}%"></div>
        </div>

        <div class="model-score-box">

            <p class="muted">
                LOGISTIC REGRESSION RAW MODEL SCORE
            </p>

            <h3 class="{{ result.model_score_class }}">
                {{ result.model_score_display }}
            </h3>

            <p class="muted">
                Raw decision_function() output (log-odds).
                Positive favours UP; negative favours DOWN.
            </p>

        </div>

        <p class="muted" style="text-align:center;">
            Model-estimated probabilities, not guaranteed outcomes.
        </p>

        <details>
            <summary>
                How are the market prediction and model score calculated?
            </summary>

            <h4>Step 1: FinBERT sentiment analysis</h4>

            FinBERT processes each headline and returns
            positive, negative and neutral class probabilities.

            <h4>Step 2: Feature aggregation</h4>

            Seven aggregated features are passed to Logistic Regression:

            <ul>
                <li>Positive mean probability</li>
                <li>Negative mean probability</li>
                <li>Neutral mean probability</li>
                <li>Positive headline ratio</li>
                <li>Negative headline ratio</li>
                <li>Neutral headline ratio</li>
                <li>Average net sentiment</li>
            </ul>

            <h4>Step 3: Feature scaling</h4>

            The features are transformed using the saved
            StandardScaler fitted on the training data.

            <h4>Step 4: Logistic Regression</h4>

            Logistic Regression produces the market prediction
            using the scaled seven-feature input.

            <div class="formula">
                Raw model score = decision_function(X_scaled)
            </div>

            A positive score favours UP, while a negative score
            favours DOWN.

            <h4>Step 5: Market probabilities</h4>

            The probabilities are obtained directly from
            Logistic Regression's predict_proba() method.

            <div class="formula">
                UP probability = predict_proba()[class 1] × 100
            </div>

            <div class="formula">
                DOWN probability = predict_proba()[class 0] × 100
            </div>

            <h4>Step 6: Prediction categories</h4>

            <ul>
                <li>DOWN: UP probability at or below 45%.</li>
                <li>MARGINALLY DOWN: above 45%, below 50%.</li>
                <li>MARGINALLY UP: 50% to below 55%.</li>
                <li>UP: 55% or higher.</li>
            </ul>

            These are prototype classification thresholds,
            not independently validated confidence levels.
        </details>

        <br>

        <div class="section-title">
            Overall News Sentiment
        </div>

        <div class="metrics">

            <div class="metric">
                <h3>{{ result.positive }}%</h3>
                <p>POSITIVE</p>
            </div>

            <div class="metric">
                <h3 style="color:#ff929e;">
                    {{ result.negative }}%
                </h3>
                <p>NEGATIVE</p>
            </div>

            <div class="metric">
                <h3>{{ result.neutral }}%</h3>
                <p>NEUTRAL</p>
            </div>

        </div>

        <p class="muted">
            These percentages show the proportion of headlines
            assigned to each dominant FinBERT sentiment category.
        </p>

    </div>

    <div class="card">

        <div class="section-title">
            Individual Headline Analysis
        </div>

        <p class="muted">
            FinBERT predictions, confidence scores and signed
            headline sentiment scores.
        </p>

        <div class="table-wrapper">

        <table>

            <thead>
                <tr>
                    <th>Headline</th>
                    <th>Sentiment</th>
                    <th>Positive</th>
                    <th>Negative</th>
                    <th>Neutral</th>
                    <th>Headline Sentiment Score</th>
                </tr>
            </thead>

            <tbody>

                {% for row in result.rows %}

                <tr>
                    <td>{{ row.headline }}</td>

                    <td class="sentiment {{ row.sentiment_class }}">
                        {{ row.sentiment }}
                    </td>

                    <td class="score">{{ row.positive }}%</td>
                    <td class="score">{{ row.negative }}%</td>
                    <td class="score">{{ row.neutral }}%</td>

                    <td class="score {{ row.net_class }}">
                        {{ row.net_score_display }}
                    </td>
                </tr>

                {% endfor %}

            </tbody>

            <tfoot>
                <tr>
                    <td colspan="5">
                        TOTAL HEADLINE SENTIMENT SCORE
                    </td>

                    <td class="{{ result.total_net_class }}">
                        {{ result.total_net_display }}
                    </td>
                </tr>
            </tfoot>

        </table>

        </div>

        <div class="net-summary">

            <div class="net-box">
                <p>TOTAL HEADLINE SENTIMENT SCORE</p>

                <h3 class="{{ result.total_net_class }}">
                    {{ result.total_net_display }}
                </h3>

                <p>Sum of individual signed FinBERT confidence scores</p>
            </div>

            <div class="net-box">
                <p>AVERAGE HEADLINE SENTIMENT SCORE</p>

                <h3 class="{{ result.average_net_class }}">
                    {{ result.average_net_display }}
                </h3>

                <p>Average signed FinBERT confidence score</p>
            </div>

        </div>

        <details>
            <summary>
                How are the headline sentiment scores calculated?
            </summary>

            <h4>Individual headline sentiment score</h4>

            FinBERT identifies the dominant sentiment and
            provides a confidence score for that prediction.

            <div class="formula">
                Positive prediction = + FinBERT confidence × 100
                <br>
                Negative prediction = - FinBERT confidence × 100
                <br>
                Neutral prediction = 0
            </div>

            For example:

            <ul>
                <li>Positive with 0.92 confidence = +92</li>
                <li>Negative with 0.85 confidence = -85</li>
                <li>Neutral with 0.95 confidence = 0</li>
            </ul>

            The confidence is the model's probability for its
            predicted class. It is not a guarantee of correctness.

            <h4>Total headline sentiment score</h4>

            The individual signed scores are added together.

            <div class="formula">
                Total score = Sum of individual scores
            </div>

            <h4>Average headline sentiment score</h4>

            <div class="formula">
                Average score = Total score / Number of headlines
            </div>

            <h4>Relationship with market prediction</h4>

            The Logistic Regression model continues to use the
            seven aggregated features.

            Its Net_Sentiment feature remains the difference
            between mean positive and mean negative FinBERT
            probabilities.

            The new signed confidence score is used for the
            individual headline display and headline summaries.
            It does not replace the original trained features.
        </details>

    </div>

    {% endif %}

</div>

<div class="footer">

    SENTRIMIX | AI-POWERED FINANCIAL SENTIMENT ANALYSIS

    <br>
    Historical training data: 2008–2016.

    <br>
    This project is a prototype and is not financial advice.

    <br>
    Predictions on current headlines have not been validated.

</div>

</body>
</html>
"""

def analyse_headlines(headlines):

    rows = []
    batch_size = 16

    with torch.inference_mode():

        for start in range(0, len(headlines), batch_size):

            batch = headlines[start:start + batch_size]

            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            )

            inputs = {
                key: value.to(device)
                for key, value in inputs.items()
            }

            outputs = finbert(**inputs)

            probabilities = torch.softmax(
                outputs.logits, dim=1
            ).cpu().numpy()

            for headline, probs in zip(batch, probabilities):

                # Get the class with the highest FinBERT probability.
                predicted_index = int(np.argmax(probs))

                predicted_label = next(
                    label for label, index in label_map.items()
                    if index == predicted_index
                )

                # Direct confidence of FinBERT's predicted class.
                confidence = float(probs[predicted_index])

                positive = float(probs[label_map["positive"]])
                negative = float(probs[label_map["negative"]])
                neutral = float(probs[label_map["neutral"]])

                # NEW SCORING METHOD:
                # Use FinBERT's predicted confidence directly.

                if predicted_label == "positive":
                    net_score = confidence * 100

                elif predicted_label == "negative":
                    net_score = -confidence * 100

                else:
                    net_score = 0.0

                rows.append({

                    "Headline": headline,

                    "Sentiment": predicted_label,

                    "Positive_Score": positive,

                    "Negative_Score": negative,

                    "Neutral_Score": neutral,

                    "Confidence": confidence,

                    "Net_Score": net_score
                })

    return pd.DataFrame(rows)


def aggregate_features(scored):

    features_dict = {

        "Positive_Mean": scored["Positive_Score"].mean(),

        "Negative_Mean": scored["Negative_Score"].mean(),

        "Neutral_Mean": scored["Neutral_Score"].mean(),

        "Positive_Ratio": (
            scored["Sentiment"] == "positive"
        ).mean(),

        "Negative_Ratio": (
            scored["Sentiment"] == "negative"
        ).mean(),

        "Neutral_Ratio": (
            scored["Sentiment"] == "neutral"
        ).mean(),

        # Keep the original feature used during training.
        "Net_Sentiment": (
            scored["Positive_Score"].mean()
            - scored["Negative_Score"].mean()
        )
    }

    return pd.DataFrame([features_dict])[features]



def format_net_score(score):

    if score > 0:
        return f"+{score:.2f}"

    return f"{score:.2f}"


def net_score_class(score):

    if score > 0.005:
        return "net-positive"

    elif score < -0.005:
        return "net-negative"

    return "net-neutral"

@app.route("/")
def home():

    return render_template_string(
        HTML_PAGE,
        result=None,
        submitted="",
        error=None
    )


@app.route("/analyse", methods=["POST"])
def analyse():

    submitted = request.form.get("headlines", "")

    headlines = [
        line.strip()
        for line in submitted.splitlines()
        if line.strip()
    ]

    if not headlines:

        return render_template_string(
            HTML_PAGE,
            result=None,
            submitted=submitted,
            error="Please enter at least one headline."
        )

    if len(headlines) > 100:

        return render_template_string(
            HTML_PAGE,
            result=None,
            submitted=submitted,
            error="Please enter no more than 100 headlines."
        )

    try:

        # FINBERT ANALYSIS

        scored = analyse_headlines(headlines)

        # AGGREGATE THE SEVEN MODEL FEATURES

        X = aggregate_features(scored)

        # SCALE FEATURES

        X_scaled = scaler.transform(X)

        # LOGISTIC REGRESSION PROBABILITIES

        probabilities = model.predict_proba(X_scaled)[0]

        classes = list(model.classes_)

        up = float(
            probabilities[classes.index(1)]
        ) * 100

        down = float(
            probabilities[classes.index(0)]
        ) * 100

        # LOGISTIC REGRESSION RAW SCORE

        if len(classes) != 2:
            raise ValueError(
                "The Logistic Regression model must be binary."
            )

        raw_score = float(
            np.asarray(
                model.decision_function(X_scaled)
            ).ravel()[0]
        )

        if classes[1] == 1:
            model_score = raw_score
        else:
            model_score = -raw_score

        # MARKET DIRECTION

        lower_threshold = math.log(0.45 / 0.55)
        upper_threshold = math.log(0.55 / 0.45)

        if model_score <= lower_threshold:

            direction = "DOWN"
            direction_class = "down"

            explanation = (
                "The model assigns a substantially higher "
                "probability to DOWN."
            )

            model_score_class = "model-negative"

        elif model_score < 0:

            direction = "MARGINALLY DOWN"
            direction_class = "marginal-down"

            explanation = (
                "The model leans slightly towards DOWN."
            )

            model_score_class = "model-amber"

        elif model_score < upper_threshold:

            direction = "MARGINALLY UP"
            direction_class = "marginal-up"

            explanation = (
                "The model leans slightly towards UP."
            )

            model_score_class = "model-amber"

        else:

            direction = "UP"
            direction_class = "up"

            explanation = (
                "The model assigns a substantially higher "
                "probability to UP."
            )

            model_score_class = "model-positive"

        model_score_display = f"{model_score:+.4f}"

        # OVERALL NEWS SENTIMENT

        positive = round(
            (scored["Sentiment"] == "positive").mean() * 100,
            1
        )

        negative = round(
            (scored["Sentiment"] == "negative").mean() * 100,
            1
        )

        neutral = round(
            (scored["Sentiment"] == "neutral").mean() * 100,
            1
        )

        # INDIVIDUAL HEADLINE RESULTS

        rows = []

        for _, row in scored.iterrows():

            sentiment = row["Sentiment"]

            net_score = float(row["Net_Score"])

            rows.append({

                "headline": row["Headline"],

                "sentiment": sentiment.title(),

                "sentiment_class": sentiment,

                "positive": round(
                    row["Positive_Score"] * 100, 1
                ),

                "negative": round(
                    row["Negative_Score"] * 100, 1
                ),

                "neutral": round(
                    row["Neutral_Score"] * 100, 1
                ),

                "net_score_display": format_net_score(
                    net_score
                ),

                "net_class": net_score_class(net_score)
            })

        # TOTAL AND AVERAGE HEADLINE SCORES

        total_net = float(
            scored["Net_Score"].sum()
        )

        average_net = float(
            scored["Net_Score"].mean()
        )

        # BUILD WEBSITE RESULTS

        result = {

            "direction": direction,

            "direction_class": direction_class,

            "explanation": explanation,

            "up": round(up, 2),

            "down": round(down, 2),

            "model_score_display": model_score_display,

            "model_score_class": model_score_class,

            "positive": positive,

            "negative": negative,

            "neutral": neutral,

            "rows": rows,

            "total_net_display": format_net_score(
                total_net
            ),

            "average_net_display": format_net_score(
                average_net
            ),

            "total_net_class": net_score_class(
                total_net
            ),

            "average_net_class": net_score_class(
                average_net
            )
        }

        return render_template_string(
            HTML_PAGE,
            result=result,
            submitted=submitted,
            error=None
        )

    except Exception as e:

        return render_template_string(
            HTML_PAGE,
            result=None,
            submitted=submitted,
            error="Analysis error: " + str(e)
        )


print("SENTRIMIX Flask Application created!")

In [ ]:
#NOTE: You will be unable to run this cell unless you have an ngrok account with a token. I have attached a video demo to my report.

from getpass import getpass
from pyngrok import ngrok

# Enter your ngrok token securely
token = getpass("Enter your ngrok authtoken: ")

# Save it in the current Colab runtime's configuration
ngrok.set_auth_token(token)

print("Ngrok authentication configured!")

In [ ]:
from pyngrok import ngrok
from threading import Thread
import time

# Use the authentication token already saved in Colab
print("Using saved ngrok authentication...")

# Close any previous tunnels
ngrok.kill()

# Start Flask in the background
server = Thread(
    target=lambda: app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    ),
    daemon=True
)

server.start()

time.sleep(3)

# Open ngrok tunnel
public_url = ngrok.connect(5000).public_url

print("====================================")
print("       SENTRIMIX IS LIVE!")
print("====================================")
print(public_url)

#This will allow you to open the link and access Sentrimix.